# Expert-CFG: test.json 생성 (RAG 전처리)

VQA-RAD의 `top_k_captions`와 `highlights` 필드가 포함된 `test.json`을 생성합니다.

**필요한 파일 (Kaggle Dataset으로 마운트):**
- `RadVQA/features/chest.pkl` 등 pkl 파일들
- `text_features.h5`
- `filter_cap.json`
- `PEIR_openai_match_keyword.json`

In [ ]:
!pip install datasets h5py -q

In [ ]:
import os

# 경로 설정 (Kaggle Dataset 마운트 위치에 맞게 수정)
DATA_ROOT = "/kaggle/input/expert-cfg-data"  # Kaggle Dataset 마운트 경로

FEATURES_DIR   = f"{DATA_ROOT}/RadVQA/features"
TEXT_FEATURES  = f"{DATA_ROOT}/text_features.h5"
FILTER_CAP     = f"{DATA_ROOT}/filter_cap.json"
MATCH_KEYWORD  = f"{DATA_ROOT}/PEIR_openai_match_keyword.json"
OUTPUT_PATH    = "/kaggle/working/test.json"
TOP_K          = 5

# 파일 존재 확인
for path in [FEATURES_DIR, TEXT_FEATURES, FILTER_CAP, MATCH_KEYWORD]:
    status = "✅" if os.path.exists(path) else "❌"
    print(f"{status} {path}")

## Step 1. pkl 파일 구조 확인

In [ ]:
import pickle

pkl_files = [f for f in os.listdir(FEATURES_DIR) if f.endswith(".pkl")]
print(f"pkl 파일: {pkl_files}")

# 첫 번째 pkl 확인
sample_pkl = os.path.join(FEATURES_DIR, pkl_files[0])
with open(sample_pkl, "rb") as f:
    data = pickle.load(f)

print(f"\n타입: {type(data)}")
if isinstance(data, dict):
    keys = list(data.keys())
    print(f"키 수: {len(keys)}")
    print(f"키 샘플: {keys[:5]}")
    sample_val = data[keys[0]]
    print(f"값 타입: {type(sample_val)}, shape: {getattr(sample_val, 'shape', 'N/A')}")
elif isinstance(data, list):
    print(f"리스트 길이: {len(data)}")
    print(f"첫 항목: {data[0]}")

## Step 2. text_features.h5 구조 확인

In [ ]:
import h5py

with h5py.File(TEXT_FEATURES, "r") as f:
    keys = list(f.keys())
    print(f"키 수: {len(keys)}")
    print(f"키 샘플: {keys[:5]}")
    sample = f[keys[0]][()]
    print(f"값 shape: {sample.shape}, dtype: {sample.dtype}")

## Step 3. VQA-RAD 데이터셋 로드 (HuggingFace)

In [ ]:
from datasets import load_dataset

ds = load_dataset("flaviagiammarino/vqa-rad", split="test")
print(f"VQA-RAD test: {len(ds)}개")
print(ds[0])

## Step 4. 피처 전체 로드

In [ ]:
import numpy as np
import json

# 이미지 피처
img_features = {}
for fname in pkl_files:
    with open(os.path.join(FEATURES_DIR, fname), "rb") as f:
        data = pickle.load(f)
    if isinstance(data, dict):
        img_features.update(data)
    elif isinstance(data, list):
        for item in data:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                img_features[item[0]] = item[1]
print(f"이미지 피처: {len(img_features)}개")

# 텍스트 피처
text_features = {}
with h5py.File(TEXT_FEATURES, "r") as f:
    for key in f.keys():
        text_features[key] = f[key][()]
text_feat_keys = list(text_features.keys())
text_feat_matrix = np.stack(list(text_features.values()), axis=0).astype(np.float32)
print(f"텍스트 피처 matrix: {text_feat_matrix.shape}")

# filter_cap
with open(FILTER_CAP) as f:
    filter_cap_raw = json.load(f)
if "annotations" in filter_cap_raw:
    filter_cap = {str(a["image_id"]): a["caption"] for a in filter_cap_raw["annotations"]}
else:
    filter_cap = {str(k): v for k, v in filter_cap_raw.items()}
print(f"filter_cap: {len(filter_cap)}개")

# match_keyword
with open(MATCH_KEYWORD) as f:
    match_keyword = json.load(f)
print(f"match_keyword: {len(match_keyword)}개")

## Step 5. test.json 생성

In [ ]:
from tqdm.notebook import tqdm

def cosine_sim(query, matrix):
    query = query / (np.linalg.norm(query) + 1e-8)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-8
    return (matrix / norms) @ query

annotations = []
skipped = 0

for item in tqdm(ds):
    img_id = str(item.get("image_name") or item.get("img_id") or item.get("image_id", ""))

    base = {
        "img_id": img_id,
        "question": item["question"],
        "answer": item["answer"],
        "answer_type": item.get("answer_type", "OPEN"),
    }

    if img_id not in img_features:
        skipped += 1
        base["top_k_captions"] = [""]
        base["highlights"] = []
        annotations.append(base)
        continue

    img_feat = np.array(img_features[img_id]).astype(np.float32).flatten()
    sims = cosine_sim(img_feat, text_feat_matrix)
    top_idx = np.argsort(sims)[::-1][:TOP_K]

    top_captions = []
    all_highlights = []

    for idx in top_idx:
        peir_key = text_feat_keys[idx]
        cap = filter_cap.get(str(peir_key), "")
        if cap:
            top_captions.append(cap)
        for cap_text, info in match_keyword.items():
            if str(peir_key) in [str(x) for x in info.get("img_ids", [])]:
                all_highlights.extend(info.get("match_keyword", []))
                break

    base["top_k_captions"] = top_captions if top_captions else [""]
    base["highlights"] = list(dict.fromkeys(all_highlights))
    annotations.append(base)

print(f"완료: {len(annotations)}개, 피처 없음: {skipped}개")

with open(OUTPUT_PATH, "w") as f:
    json.dump(annotations, f, indent=2, ensure_ascii=False)
print(f"저장: {OUTPUT_PATH}")

## Step 6. 결과 확인

In [ ]:
sample = annotations[0]
print(f"img_id: {sample['img_id']}")
print(f"question: {sample['question']}")
print(f"answer: {sample['answer']}")
print(f"top_k_captions[0]: {sample['top_k_captions'][0][:100]}")
print(f"highlights: {sample['highlights'][:5]}")
print(f"\n✅ test.json 생성 완료 → {OUTPUT_PATH}")